In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = "finalyear-project-rev2"
REPO_URL = "https://github.com/AmarjithTK/finalyear-project-rev2.git"

try:
    import google.colab  # type: ignore
    colabrun = True
except ImportError:
    colabrun = False

if colabrun:
    repo_path = Path("/content") / REPO
    if repo_path.exists():
        shutil.rmtree(repo_path)

    subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
    os.chdir(repo_path)
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "xgboost"], check=True)
else:
    repo_path = Path.cwd()

basedir = str(repo_path) + os.sep
data_path = os.path.join(basedir, "data", "microgrid.csv")

print("Running in Colab:", colabrun)
print("Current directory:", os.getcwd())
print("Base directory set to:", basedir)
print("Data path:", data_path)

In [ ]:
import pandas as pd
import os

# Construct the file path using the basedir defined in the previous cell
data_path = os.path.join(basedir, 'data/microgrid.csv')

try:
    # Read the CSV file and display the first few rows
    df = pd.read_csv(data_path)
    display(df.head())
except Exception as e:
    print(f"Failed to read CSV at {data_path}: {e}")

In [ ]:
# 1. Execute Training & Prediction Scripts
# Handled via the .py scripts in src/
# If in Colab, this will execute the cloned python modules.
# Optional dependency install in Colab:
# !pip install -r {basedir}requirements.txt

!cd {basedir} && python -m src.train
!cd {basedir} && python -m src.predict
!cd {basedir} && python -m src.scenarios

In [ ]:
# 2. Load Outputs and Build Error Tables
import json
import numpy as np

try:
    preds = pd.read_csv(os.path.join(basedir, "outputs/predictions.csv"))
    preds["timestamp"] = pd.to_datetime(preds["timestamp"])

    with open(os.path.join(basedir, "outputs/metrics.json")) as f:
        metrics = json.load(f)

    with open(os.path.join(basedir, "outputs/scenarios.json")) as f:
        scenarios = json.load(f)

    pred_cols = [c for c in preds.columns if c.startswith("predicted_")]
    target_names = [c.replace("predicted_", "") for c in pred_cols]

    for target in target_names:
        actual_col = f"actual_{target}"
        pred_col = f"predicted_{target}"
        if actual_col in preds.columns:
            preds[f"error_{target}"] = preds[pred_col] - preds[actual_col]
            preds[f"abs_error_{target}"] = preds[f"error_{target}"].abs()
            preds[f"squared_error_{target}"] = preds[f"error_{target}"] ** 2
            preds[f"pct_error_{target}"] = np.where(
                preds[actual_col] != 0,
                100 * preds[f"error_{target}"] / preds[actual_col],
                np.nan,
            )
            preds[f"abs_pct_error_{target}"] = preds[f"pct_error_{target}"].abs()

    if {"predicted_solar_pv_output", "predicted_wind_power_output"}.issubset(preds.columns):
        preds["total_predicted_energy"] = preds["predicted_solar_pv_output"] + preds["predicted_wind_power_output"]

    if {"actual_solar_pv_output", "actual_wind_power_output"}.issubset(preds.columns):
        preds["actual_total_renewable_energy"] = preds["actual_solar_pv_output"] + preds["actual_wind_power_output"]

    if {"total_predicted_energy", "predicted_grid_load_demand"}.issubset(preds.columns):
        preds["net_grid_balance_predicted"] = preds["total_predicted_energy"] - preds["predicted_grid_load_demand"]
        preds["grid_import_predicted"] = (-preds["net_grid_balance_predicted"]).clip(lower=0)
        preds["grid_export_predicted"] = preds["net_grid_balance_predicted"].clip(lower=0)

    if {"actual_total_renewable_energy", "actual_grid_load_demand"}.issubset(preds.columns):
        preds["net_grid_balance_actual"] = preds["actual_total_renewable_energy"] - preds["actual_grid_load_demand"]

    computed_metric_rows = []
    for target in target_names:
        actual_col = f"actual_{target}"
        pred_col = f"predicted_{target}"
        error_col = f"error_{target}"
        abs_error_col = f"abs_error_{target}"
        squared_error_col = f"squared_error_{target}"
        if actual_col in preds.columns:
            actual = preds[actual_col]
            predicted = preds[pred_col]
            abs_error = preds[abs_error_col]
            squared_error = preds[squared_error_col]
            mean_actual = actual.abs().mean()
            actual_range = actual.max() - actual.min()
            mae = abs_error.mean()
            rmse = np.sqrt(squared_error.mean())
            nonzero_actual = actual != 0
            smape_denominator = actual.abs() + predicted.abs()
            r2_denominator = ((actual - actual.mean()) ** 2).sum()
            computed_metric_rows.append({
                "target": target,
                "MAE_%_of_mean": (mae / mean_actual) * 100 if mean_actual else np.nan,
                "RMSE_%_of_mean": (rmse / mean_actual) * 100 if mean_actual else np.nan,
                "NRMSE_%_of_range": (rmse / actual_range) * 100 if actual_range else np.nan,
                "MAPE_%": (100 * abs_error[nonzero_actual] / actual[nonzero_actual].abs()).mean(),
                "sMAPE_%": (200 * abs_error[smape_denominator != 0] / smape_denominator[smape_denominator != 0]).mean(),
                "Bias_%_of_mean": (preds[error_col].mean() / mean_actual) * 100 if mean_actual else np.nan,
                "R2": 1 - (squared_error.sum() / r2_denominator) if r2_denominator else np.nan,
                "MAE": abs_error.mean(),
                "MSE": squared_error.mean(),
                "RMSE": np.sqrt(squared_error.mean()),
                "MedianAE": abs_error.median(),
                "MaxAE": abs_error.max(),
                "MAPE_percent": (100 * abs_error[nonzero_actual] / actual[nonzero_actual].abs()).mean(),
                "sMAPE_percent": (200 * abs_error[smape_denominator != 0] / smape_denominator[smape_denominator != 0]).mean(),
                "Mean_Bias_Error": preds[error_col].mean(),
                "Error_STD": preds[error_col].std(),
            })

    computed_metrics_df = pd.DataFrame(computed_metric_rows)
    if not computed_metrics_df.empty:
        computed_metrics_df = computed_metrics_df.set_index("target").round(4)
    saved_metrics_df = pd.DataFrame(metrics).T
    saved_numeric_cols = saved_metrics_df.select_dtypes(include="number").columns
    saved_metrics_df[saved_numeric_cols] = saved_metrics_df[saved_numeric_cols].round(4)

    print("Files loaded successfully")
    print(f"Prediction rows: {len(preds):,}")
    print(f"Detected targets: {target_names}")

    percent_metric_cols = [
        "MAE_%_of_mean",
        "RMSE_%_of_mean",
        "NRMSE_%_of_range",
        "MAPE_%",
        "sMAPE_%",
        "Bias_%_of_mean",
        "R2",
    ]
    raw_metric_cols = ["MAE", "MSE", "RMSE", "MedianAE", "MaxAE", "Mean_Bias_Error", "Error_STD"]

    print("Primary relative metrics")
    display(computed_metrics_df[[c for c in percent_metric_cols if c in computed_metrics_df.columns]])
    print("Secondary raw-unit metrics")
    display(computed_metrics_df[[c for c in raw_metric_cols if c in computed_metrics_df.columns]])
    print("Saved metrics from outputs/metrics.json")
    display(saved_metrics_df)
    display(preds.head(10))
    display(preds.tail(10))
except Exception as e:
    print(f"Failed to load outputs: {e}")

In [ ]:
# 3. Expanded Visualizations
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

if 'preds' in locals():
    pred_cols = [c for c in preds.columns if c.startswith("predicted_")]
    if not pred_cols:
        print("No predicted_* columns found in outputs/predictions.csv")
    else:
        for pred_col in pred_cols:
            target = pred_col.replace("predicted_", "")
            actual_col = f"actual_{target}"
            error_col = f"error_{target}"
            abs_error_col = f"abs_error_{target}"
            squared_error_col = f"squared_error_{target}"
            pct_error_col = f"pct_error_{target}"
            abs_pct_error_col = f"abs_pct_error_{target}"

            if actual_col not in preds.columns:
                plt.figure(figsize=(18, 6))
                plt.plot(preds["timestamp"], preds[pred_col], label=f"Predicted {target}", linewidth=2)
                plt.title(f"Predicted {target}", fontsize=16)
                plt.xlabel("Time")
                plt.ylabel(target)
                plt.legend()
                plt.show()
                continue

            fig, axes = plt.subplots(4, 2, figsize=(22, 24))
            fig.suptitle(f"Detailed Prediction Diagnostics: {target}", fontsize=20, y=0.995)

            axes[0, 0].plot(preds["timestamp"], preds[actual_col], label="Actual", linewidth=2)
            axes[0, 0].plot(preds["timestamp"], preds[pred_col], label="Predicted", linewidth=2, alpha=0.85)
            axes[0, 0].set_title("Actual vs Predicted", fontsize=14)
            axes[0, 0].set_xlabel("Time")
            axes[0, 0].set_ylabel(target)
            axes[0, 0].legend()

            axes[0, 1].plot(preds["timestamp"], preds[pct_error_col], color="tab:red", linewidth=1.8)
            axes[0, 1].axhline(0, color="black", linewidth=1)
            axes[0, 1].set_title("Percentage Error Over Time", fontsize=14)
            axes[0, 1].set_xlabel("Time")
            axes[0, 1].set_ylabel("Error (%)")

            axes[1, 0].scatter(preds[actual_col], preds[pred_col], alpha=0.65)
            min_val = min(preds[actual_col].min(), preds[pred_col].min())
            max_val = max(preds[actual_col].max(), preds[pred_col].max())
            axes[1, 0].plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--")
            axes[1, 0].set_title("Actual vs Predicted Scatter", fontsize=14)
            axes[1, 0].set_xlabel("Actual")
            axes[1, 0].set_ylabel("Predicted")

            axes[1, 1].hist(preds[pct_error_col].dropna(), bins=35, color="tab:purple", alpha=0.75)
            axes[1, 1].axvline(0, color="black", linewidth=1)
            axes[1, 1].set_title("Percentage Error Distribution", fontsize=14)
            axes[1, 1].set_xlabel("Error (%)")
            axes[1, 1].set_ylabel("Count")

            rolling_window = max(3, min(24, len(preds) // 5))
            rolling_mape = preds[abs_pct_error_col].rolling(rolling_window, min_periods=1).mean()
            axes[2, 0].plot(preds["timestamp"], rolling_mape, color="tab:orange", linewidth=2)
            axes[2, 0].set_title(f"Rolling Absolute Percentage Error ({rolling_window} points)", fontsize=14)
            axes[2, 0].set_xlabel("Time")
            axes[2, 0].set_ylabel("Absolute Error (%)")

            axes[2, 1].plot(preds["timestamp"], preds[abs_pct_error_col], color="tab:green", linewidth=1.8)
            axes[2, 1].set_title("Absolute Percentage Error Over Time", fontsize=14)
            axes[2, 1].set_xlabel("Time")
            axes[2, 1].set_ylabel("Absolute Error (%)")

            axes[3, 0].boxplot(preds[abs_pct_error_col].dropna(), vert=False)
            axes[3, 0].set_title("Absolute Percentage Error Spread", fontsize=14)
            axes[3, 0].set_xlabel("Absolute Error (%)")

            cumulative_bias = preds[pct_error_col].fillna(0).cumsum()
            axes[3, 1].plot(preds["timestamp"], cumulative_bias, color="tab:brown", linewidth=2)
            axes[3, 1].axhline(0, color="black", linewidth=1)
            axes[3, 1].set_title("Cumulative Signed Percentage Error", fontsize=14)
            axes[3, 1].set_xlabel("Time")
            axes[3, 1].set_ylabel("Cumulative Error (%)")

            plt.tight_layout()
            plt.show()

            error_summary = preds[[pct_error_col, abs_pct_error_col, error_col, abs_error_col]].describe().T.round(4)
            display(error_summary)

            worst_cols = ["timestamp", actual_col, pred_col, pct_error_col, abs_pct_error_col, error_col]
            worst_rows = preds[worst_cols].sort_values(abs_pct_error_col, ascending=False).head(10).round(4)
            print(f"Worst 10 relative-error rows for {target}")
            display(worst_rows)

            hourly_error = preds.assign(hour=preds["timestamp"].dt.hour).groupby("hour")[abs_pct_error_col].mean()
            if not hourly_error.empty:
                plt.figure(figsize=(14, 5))
                plt.bar(hourly_error.index, hourly_error.values, color="tab:cyan")
                plt.title(f"Mean Absolute Percentage Error by Hour: {target}", fontsize=15)
                plt.xlabel("Hour of Day")
                plt.ylabel("Mean Absolute Error (%)")
                plt.xticks(range(0, 24))
                plt.tight_layout()
                plt.show()

            heatmap_data = preds.assign(
                day_of_week=preds["timestamp"].dt.dayofweek,
                hour=preds["timestamp"].dt.hour,
            ).pivot_table(values=abs_pct_error_col, index="day_of_week", columns="hour", aggfunc="mean")
            if not heatmap_data.empty:
                heatmap_data = heatmap_data.reindex(index=range(7), columns=range(24))
                plt.figure(figsize=(16, 5))
                im = plt.imshow(heatmap_data, aspect="auto", cmap="YlOrRd")
                plt.colorbar(im, label="Mean Absolute Error (%)")
                plt.title(f"Absolute Percentage Error Heatmap: {target}", fontsize=15)
                plt.xlabel("Hour of Day")
                plt.ylabel("Day of Week")
                plt.xticks(range(24))
                plt.yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
                plt.tight_layout()
                plt.show()

        required_energy_cols = [
            "predicted_grid_load_demand",
            "predicted_solar_pv_output",
            "predicted_wind_power_output",
            "total_predicted_energy",
            "net_grid_balance_predicted",
            "grid_import_predicted",
            "grid_export_predicted",
        ]
        if all(col in preds.columns for col in required_energy_cols):
            fig, axes = plt.subplots(3, 1, figsize=(22, 18), sharex=True)
            fig.suptitle("Forecasted Renewable Generation, Load, and Grid Exchange", fontsize=20, y=0.995)

            axes[0].stackplot(
                preds["timestamp"],
                preds["predicted_solar_pv_output"],
                preds["predicted_wind_power_output"],
                labels=["Predicted Solar PV", "Predicted Wind"],
                alpha=0.8,
            )
            axes[0].plot(preds["timestamp"], preds["total_predicted_energy"], color="black", linewidth=2, label="Total predicted renewable")
            axes[0].set_ylabel("Power")
            axes[0].set_title("Predicted Renewable Generation Components", fontsize=14)
            axes[0].legend(loc="upper left")

            axes[1].plot(preds["timestamp"], preds["predicted_grid_load_demand"], label="Predicted load demand", linewidth=2)
            axes[1].plot(preds["timestamp"], preds["total_predicted_energy"], label="Total predicted renewable", linewidth=2)
            axes[1].set_ylabel("Power")
            axes[1].set_title("Predicted Load vs Renewable Supply", fontsize=14)
            axes[1].legend(loc="upper left")

            axes[2].plot(preds["timestamp"], preds["net_grid_balance_predicted"], color="tab:brown", label="Renewable - Load", linewidth=2)
            axes[2].fill_between(preds["timestamp"], 0, preds["grid_import_predicted"], alpha=0.35, label="Grid import need")
            axes[2].fill_between(preds["timestamp"], 0, preds["grid_export_predicted"], alpha=0.35, label="Grid export surplus")
            axes[2].axhline(0, color="black", linewidth=1)
            axes[2].set_title("Predicted Grid Import / Export Balance", fontsize=14)
            axes[2].set_xlabel("Time")
            axes[2].set_ylabel("Power")
            axes[2].legend(loc="upper left")

            plt.tight_layout()
            plt.show()

            display(preds[["timestamp"] + required_energy_cols].head(24).round(4))

In [ ]:
# 4. Scenario Assessment
if 'scenarios' in locals():
    scenario_rows = []
    for scenario_name, details in scenarios.items():
        row = {"scenario": scenario_name.replace("_", " ")}
        row.update(details)
        scenario_rows.append(row)

    scenarios_df = pd.DataFrame(scenario_rows)
    display(scenarios_df)

    if "count" in scenarios_df.columns:
        plt.figure(figsize=(14, 5))
        plt.bar(scenarios_df["scenario"], scenarios_df["count"].fillna(0))
        plt.title("Scenario Event Counts", fontsize=15)
        plt.xlabel("Scenario")
        plt.ylabel("Event Count")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()